# Prompt 버전 관리와 A/B 테스트

프롬프트가 여러 번 수정되면서 의도가 흐려지는 **Prompt Drift** 현상을 방지하기 위해,
프롬프트를 코드처럼 **SemVer**로 버전 관리하고 **A/B 테스트**로 검증하는 방법을 실습한다.

### 학습 목표

1. 프롬프트를 딕셔너리 기반 Registry로 관리하는 패턴을 익힌다
2. LM Judge로 프롬프트 버전 간 성능을 정량 비교한다
3. 해시 기반 A/B 테스트로 트래픽을 분배하는 로직을 구현한다
4. Langfuse에서 버전별 트레이스를 필터링하고 비교한다

---

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler
import hashlib

## 1. Prompt Registry — 프롬프트를 코드처럼 관리하기

프롬프트도 코드다. 변경할 때마다 **SemVer** 규칙으로 버전을 매기고, 변경 이유를 기록한다.

| 변경 유형 | 버전 증가 | 예시 |
|-----------|----------|------|
| 형식만 수정 | Patch (v1.0.0→v1.0.1) | 줄바꿈 조정 |
| 동작 개선 | Minor (v1.0.0→v1.1.0) | 예시 추가 |
| 역할·목적 변경 | Major (v1.0.0→v2.0.0) | 시스템 프롬프트 전면 교체 |

In [2]:
# ── Prompt Registry 구현 ──
PROMPT_REGISTRY: dict[str, dict] = {}


def register_prompt(name: str, version: str, content: str, changelog: str = ""):
    """프롬프트를 레지스트리에 등록한다."""
    key = f"{name}@{version}"
    PROMPT_REGISTRY[key] = {
        "name": name,
        "version": version,
        "content": content,
        "changelog": changelog,
    }
    print(f"✅ 등록: {key}")


def get_prompt(name: str, version: str) -> str:
    """특정 버전의 프롬프트를 가져온다."""
    return PROMPT_REGISTRY[f"{name}@{version}"]["content"]


def list_versions(name: str):
    """프롬프트의 모든 버전을 출력한다."""
    versions = [v for v in PROMPT_REGISTRY.values() if v["name"] == name]
    for v in sorted(versions, key=lambda x: x["version"]):
        print(f"  {v['version']}: {v['changelog']}")

In [3]:
# ── 고객 상담 프롬프트 3개 버전 등록 ──
# 당장의 성능 보다도 향후 버전별 성능 판단을 하고, 버전별로 어떤 변화가 있었는지 추적하는 것이 중요하기 때문에, 의도적으로 작은 변화부터 큰 변화까지 다양한 버전을 등록한다.
register_prompt(
    name="customer-support",
    version="v1.0.0",
    content="당신은 고객 상담원입니다. 고객의 질문에 답변하세요.",
    changelog="초기 버전 — 최소한의 지시만 포함",
)

register_prompt(
    name="customer-support",
    version="v1.1.0",
    content="""당신은 전자상거래 플랫폼의 고객 상담원입니다.

규칙:
- 항상 공손하고 전문적인 어조로 답변합니다.
- 불확실한 내용은 추측하지 않고 "확인 후 안내드리겠습니다"라고 답합니다.
- 답변 마지막에 추가 도움이 필요한지 묻습니다.""",
    changelog="톤 가이드 + 불확실성 처리 규칙 추가",
)

register_prompt(
    name="customer-support",
    version="v2.0.0",
    content="""당신은 **ShopEasy** 전자상거래 플랫폼의 시니어 고객 상담원입니다.

## 답변 형식
1. 고객 문의를 한 문장으로 요약
2. 구체적 해결 방안을 단계별로 안내
3. 추가 문의 여부 확인

## 정책
- 환불: 구매 후 14일 이내, 미개봉 상품만 가능
- 교환: 구매 후 30일 이내, 동일 가격대 상품
- 배송: 서울 1~2일, 지방 2~3일

## 주의사항
- 정책 범위를 벗어나는 요청 → "담당 부서에 확인 후 안내드리겠습니다"
- 개인정보(카드번호, 비밀번호)를 절대 요청하지 않음""",
    changelog="전면 개편 — 구조화된 형식 + 구체적 정책 포함",
)

print("\n📋 등록된 버전:")
list_versions("customer-support")

✅ 등록: customer-support@v1.0.0
✅ 등록: customer-support@v1.1.0
✅ 등록: customer-support@v2.0.0

📋 등록된 버전:
  v1.0.0: 초기 버전 — 최소한의 지시만 포함
  v1.1.0: 톤 가이드 + 불확실성 처리 규칙 추가
  v2.0.0: 전면 개편 — 구조화된 형식 + 구체적 정책 포함


## 2. 버전별 응답 비교

같은 질문에 대해 각 버전이 어떻게 다르게 응답하는지 확인한다.
Langfuse에 `langfuse_tags`로 버전을 태깅하여 나중에 필터링할 수 있게 한다.

In [ ]:
test_questions = [
    "배송이 너무 늦어요. 서울인데 주문한 지 5일 됐어요.",
    "이 상품 환불하고 싶은데 3주 전에 샀어요.",
    "제 카드번호 알려드릴 테니 직접 결제 처리해주세요.",
]

versions = ["v1.0.0", "v1.1.0", "v2.0.0"]  # v3.0.0은 존재하지 않는 버전으로, 프롬프트가 없을 때 모델이 어떻게 반응하는지 테스트하기 위해 포함한다.
results = {}  # {version: [{question, answer}, ...]}

for version in versions:
    system_prompt = get_prompt("customer-support", version)
    llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
    handler = CallbackHandler()

    results[version] = []
    for q in test_questions:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": q},
        ]
        response = llm.invoke(
            messages,
            config={
                "callbacks": [handler],
                "metadata": {
                    #Langfuse에 프롬프트 버전 정보를 태그로 남겨서, 나중에 대시보드에서 버전별 성능을 쉽게 비교할 수 있도록 한다.
                    #Navigator : Langfuse > Tags > prompt-v1.0.0, prompt-v1.1.0, prompt-v2.0.0 태그로 버전별 성능 비교 가능
                    "langfuse_tags": [f"prompt-{version}", "versioning-lab"],
                    #하나의 세션에서 여러 버전을 테스트하더라도, 동일한 세션 ID를 사용해서 버전별 성능을 한눈에 비교할 수 있도록 한다.
                    #Navigator : Langfuse > Sessions > prompt-versioning-lab 세션에서 버전별 성능 비교 가능
                    "langfuse_session_id": "prompt-versioning-lab",
                    "prompt_version": version,
                },
            },
        )
        results[version].append({"question": q, "answer": response.content})
        print(f"[{version}] Q: {q[:30]}... ✓")

[v1.0.0] Q: 배송이 너무 늦어요. 서울인데 주문한 지 5일 됐어요.... ✓
[v1.0.0] Q: 이 상품 환불하고 싶은데 3주 전에 샀어요.... ✓
[v1.0.0] Q: 제 카드번호 알려드릴 테니 직접 결제 처리해주세요.... ✓
[v1.1.0] Q: 배송이 너무 늦어요. 서울인데 주문한 지 5일 됐어요.... ✓
[v1.1.0] Q: 이 상품 환불하고 싶은데 3주 전에 샀어요.... ✓
[v1.1.0] Q: 제 카드번호 알려드릴 테니 직접 결제 처리해주세요.... ✓
[v2.0.0] Q: 배송이 너무 늦어요. 서울인데 주문한 지 5일 됐어요.... ✓
[v2.0.0] Q: 이 상품 환불하고 싶은데 3주 전에 샀어요.... ✓
[v2.0.0] Q: 제 카드번호 알려드릴 테니 직접 결제 처리해주세요.... ✓


In [15]:
# ── 응답 비교 ──
for q_idx, q in enumerate(test_questions):
    print(f"\n{'=' * 70}")
    print(f"Q: {q}")
    for version in versions:
        answer = results[version][q_idx]["answer"]
        preview = answer[:200] + "..." if len(answer) > 200 else answer
        print(f"\n--- {version} ---")
        print(preview)


Q: 배송이 너무 늦어요. 서울인데 주문한 지 5일 됐어요.

--- v1.0.0 ---
불편을 드려 죄송합니다. 서울이면 보통 더 빠르게 받아보셔야 하는데, 5일이나 지연되면 확인이 필요합니다.

확인을 위해 아래 정보 중 하나를 알려주세요:
- 주문번호
- 받는 분 성함
- 연락처 뒤 4자리

바로 배송 상태를 확인해드리겠습니다.  
원하시면 제가 먼저 **지연 사유 확인용 안내 문구**도 드릴게요.

--- v1.1.0 ---
불편을 드려 죄송합니다. 서울 지역인데도 주문 후 5일이 지났다면 배송이 지연된 것으로 보입니다.  
정확한 배송 상태를 확인해 보아야 해서, 주문번호를 알려주시면 바로 확인 후 안내드리겠습니다.

추가로 도움이 필요하신가요?

--- v2.0.0 ---
1. **문의 요약**  
서울 지역 배송이 주문 후 5일째 지연되고 있어 확인이 필요하다는 문의입니다.

2. **구체적 해결 방안**  
- ShopEasy의 서울 배송 기준은 **1~2일**입니다. 현재 주문 후 **5일 경과**라면 정상 배송 범위를 벗어난 상태입니다.  
- 먼저 주문 내역에서 **배송 상태(출고 완료/배송 중/배송사 인계 여부)*...

Q: 이 상품 환불하고 싶은데 3주 전에 샀어요.

--- v1.0.0 ---
환불을 원하시는군요.  
보통 **구매 후 3주**가 지났다면 상품 종류와 상태에 따라 환불 가능 여부가 달라집니다.

확인해볼 사항:
1. **구매처 환불 정책**: 대부분 교환/환불 가능 기간이 정해져 있습니다.
2. **상품 상태**: 미개봉/미사용인지, 포장 훼손 여부
3. **상품 종류**: 일부 상품은 환불이 제한될 수 있습니다.  
   - 식...

--- v1.1.0 ---
환불 원하시는군요. 도와드리겠습니다.

구매하신 지 3주가 지난 경우, 상품 종류와 상태, 그리고 판매처의 환불 정책에 따라 환불 가능 여부가 달라질 수 있습니다. 정확한 확인이 필요해서 **확인 후 안내드리겠습니다**.

가능하면 아래 정보를 알려주세요.

## 3. LM Judge로 정량 비교

사람이 모든 응답을 일일이 평가할 수 없다.
**LLM을 평가자(Judge)로 활용**하여 두 버전의 응답을 비교한다.

In [16]:
def pairwise_judge(question: str, answer_a: str, answer_b: str, label_a: str, label_b: str) -> dict:
    """두 응답을 비교하여 승자를 판정한다."""
    judge = ChatOpenAI(model="gpt-5.4", temperature=0)
    handler = CallbackHandler()

    prompt = f"""두 고객 상담 응답을 비교 평가하세요.

[질문]
{question}

[A 응답]
{answer_a}

[B 응답]
{answer_b}

평가 기준:
1. 정확성 — 정책에 맞는 답변인가
2. 전문성 — 어조가 적절한가
3. 완결성 — 필요한 정보를 모두 제공하는가

반드시 아래 형식으로만 답하세요:
승자: A 또는 B 또는 TIE
이유: (한 문장)"""

    response = judge.invoke(
        prompt,
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": ["lm-judge", "versioning-lab"],
                "comparison": f"{label_a}_vs_{label_b}",
            },
        },
    )
    return {"comparison": f"{label_a} vs {label_b}", "result": response.content}

In [7]:
# ── v1.0.0 vs v2.0.0 Pairwise 비교 ──
print("📊 LM Judge 평가: v1.0.0 vs v2.0.0\n")

for q_idx, q in enumerate(test_questions):
    a = results["v1.0.0"][q_idx]["answer"]
    b = results["v2.0.0"][q_idx]["answer"]
    verdict = pairwise_judge(q, a, b, "v1.0.0", "v2.0.0")
    print(f"Q: {q[:50]}...")
    print(f"   {verdict['result']}\n")

📊 LM Judge 평가: v1.0.0 vs v2.0.0

Q: 배송이 너무 늦어요. 서울인데 주문한 지 5일 됐어요....
   승자: A
이유: 사과와 공감이 자연스럽고 확인에 필요한 정보를 구체적으로 요청하며 지연 사유와 예상 배송일까지 안내하겠다고 해 정확성·전문성·완결성의 균형이 가장 좋습니다.

Q: 이 상품 환불하고 싶은데 3주 전에 샀어요....
   승자: A
이유: A는 구체적 정책이 확인되지 않은 상황에서 가정하지 않고 필요한 확인 정보와 판단 기준을 함께 안내해 정확성·전문성·완결성이 더 높고, B는 근거 없이 특정 환불 정책을 단정해 부정확할 수 있습니다.

Q: 제 카드번호 알려드릴 테니 직접 결제 처리해주세요....
   승자: B
이유: 카드정보 수집·대리 결제를 명확히 거절하면서도 안전한 대안, 오류 대응, 이상 결제 후속 안내까지 포함해 정확성·전문성·완결성이 더 높습니다.



## 4. A/B 테스트 트래픽 분배

프롬프트를 바꾸기 전에 A/B 테스트로 검증한다.
사용자 ID의 해시값으로 트래픽을 균등 분배하면, **같은 사용자는 항상 같은 버전**을 받는다.

In [8]:
def get_prompt_version_ab(user_id: str, versions: list[str] = ["v1.1.0", "v2.0.0"]) -> str:
    """user_id의 해시값으로 버전을 결정한다. 같은 유저는 항상 같은 버전을 받는다."""
    hash_val = int(hashlib.md5(user_id.encode()).hexdigest(), 16)
    return versions[hash_val % len(versions)]


# ── 1000명 시뮬레이션 ──
from collections import Counter

assignments = Counter()
for i in range(1000):
    version = get_prompt_version_ab(f"user-{i}")
    assignments[version] += 1

print("🎯 A/B 트래픽 분배 결과 (1000명 시뮬레이션):")
for version, count in sorted(assignments.items()):
    print(f"  {version}: {count}명 ({count / 10:.1f}%)")

🎯 A/B 트래픽 분배 결과 (1000명 시뮬레이션):
  v1.1.0: 501명 (50.1%)
  v2.0.0: 499명 (49.9%)


In [9]:
# ── A/B 테스트 실제 실행 ──
# 5명의 가상 유저가 같은 질문을 했을 때, 버전별 응답 비교

sample_users = ["user-42", "user-77", "user-128", "user-256", "user-999"]
question = "환불 기한이 지났는데 환불 가능한가요?"
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
handler = CallbackHandler()

print(f"Q: {question}\n")
for user_id in sample_users:
    version = get_prompt_version_ab(user_id)
    system_prompt = get_prompt("customer-support", version)

    response = llm.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": [f"ab-test", f"prompt-{version}"],
                "langfuse_user_id": user_id,
                "langfuse_session_id": "ab-test-lab",
                "prompt_version": version,
            },
        },
    )
    preview = response.content[:120] + "..." if len(response.content) > 120 else response.content
    print(f"  [{user_id}] → {version}: {preview}")

Q: 환불 기한이 지났는데 환불 가능한가요?

  [user-42] → v2.0.0: 1. **문의 요약:** 환불 기한이 지난 경우에도 환불이 가능한지 확인 요청하셨습니다.

2. **해결 방안:**  
   - ShopEasy 환불 정책상 **구매 후 14일 이내**이고 **미개봉 상품**인 경우...
  [user-77] → v1.1.0: 안녕하세요, 고객님. 환불 기한이 지난 경우에는 일반적으로 즉시 환불이 어려울 수 있습니다. 다만 주문 상태, 상품 유형, 결제 수단, 예외 규정에 따라 가능 여부가 달라질 수 있어 확인이 필요합니다.

정확한 안내...
  [user-128] → v1.1.0: 안녕하세요, 고객님.

환불 기한이 지난 경우에는 일반적으로 환불이 어려울 수 있습니다. 다만 상품 상태, 주문 내역, 결제 방식, 내부 정책에 따라 예외적으로 확인이 필요한 경우가 있어 **확인 후 안내드리겠습니다...
  [user-256] → v1.1.0: 안녕하세요, 고객님.  
환불 기한이 지난 경우에는 일반적으로 환불이 제한될 수 있습니다. 다만 상품 상태, 구매 사유, 판매자 정책, 그리고 예외 적용 여부에 따라 가능할 수도 있어 확인이 필요합니다.

정확한 안...
  [user-999] → v2.0.0: 1. **문의 요약**  
   환불 기한이 지난 경우에도 환불이 가능한지 문의하셨습니다.

2. **구체적 안내**  
   ShopEasy 환불 정책상 **구매 후 14일 이내**이면서 **미개봉 상품만 환불 가...


## 5. Langfuse에서 확인할 포인트

Langfuse UI(`http://localhost:3000`)에서 다음을 확인해보세요:

1. **Traces → Tags 필터**: `prompt-v1.0.0`, `prompt-v2.0.0` 등으로 필터링하여 버전별 트레이스 비교
2. **Sessions**: `prompt-versioning-lab`, `ab-test-lab` 세션으로 그룹화된 트레이스 확인
3. **Metadata**: 각 트레이스의 `prompt_version` 메타데이터 확인
4. **LM Judge 트레이스**: `lm-judge` 태그로 필터링하여 평가 과정 확인
5. **Latency/Cost**: 프롬프트 길이에 따른 응답 시간과 비용 차이 비교

---

## 🔧 실습: 프롬프트 v3.0.0 작성 및 테스트

**목표**: v2.0.0의 약점을 보완하는 v3.0.0 프롬프트를 작성하고 LM Judge로 비교한다.

**가이드**:
- v2.0.0 응답에서 개선할 점을 찾는다 (예: 공감 표현 부족, 대안 제시 미흡)
- v3.0.0에 개선 사항을 반영한다
- 동일한 테스트 질문으로 평가한다

In [17]:
# TODO: v3.0.0 프롬프트를 작성하세요
register_prompt(
    name="customer-support",
    version="v3.0.0",
    content="""당신은 **ShopEasy** 전자상거래 플랫폼의 베테랑 고객 상담원입니다.

## 답변 형식
1. 고객 문의를 한 문장으로 요약
2. 구체적 해결 방안을 단계별로 안내
3. 추가 문의 여부 확인
4. 불만인 고객의 공감을 표현하고, 친절한 마무리 인사 포함한다.

## 정책
- 환불: 구매 후 14일 이내, 미개봉 상품만 가능
- 교환: 구매 후 30일 이내, 동일 가격대 상품
- 배송: 서울 1~2일, 지방 2~3일, 도서 산간 지역은 추가 2일, 해외는 배송이 불가능

## 주의사항
- 정책 범위를 벗어나는 요청 → "담당 부서에 확인 후 안내드리겠습니다"
- 개인정보(카드번호, 비밀번호)를 절대 요청하지 않음""",
    changelog= "부분 개편 — 구조화된 형식 + 구체적 정책 포함",
)

# TODO: 테스트 실행
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
handler = CallbackHandler()

for q in test_questions:
    system_prompt = get_prompt("customer-support", "v3.0.0")
    response = llm.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": q},
        ],
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": ["prompt-v3.0.0", "versioning-lab"],
                "prompt_version": "v3.0.0",
            },
        },
    )
    print(f"Q: {q[:40]}...")
    print(f"A: {response.content[:200]}\n")

✅ 등록: customer-support@v3.0.0
Q: 배송이 너무 늦어요. 서울인데 주문한 지 5일 됐어요....
A: 1. **고객 문의 요약**  
   서울 지역인데 주문 후 5일이 지났는데도 배송이 지연되고 있다는 문의로 이해했습니다.

2. **해결 방안 안내**  
   - 서울 지역 배송은 보통 **1~2일** 소요됩니다.  
   - 다만 현재 5일째 미도착이라면 **배송 지연 가능성**이 있어 확인이 필요합니다.  
   - 주문 상태를 확인해 보시고, 배송

Q: 이 상품 환불하고 싶은데 3주 전에 샀어요....
A: 1. 고객 문의 요약:  
구매하신 상품을 3주 전에 구매하셨는데 환불 가능 여부를 문의하셨습니다.

2. 해결 방안 안내:  
- ShopEasy 환불 정책상 **구매 후 14일 이내**이면서 **미개봉 상품**만 환불이 가능합니다.  
- 고객님께서는 **3주 전 구매**로 확인되어, 현재는 환불 가능 기간을 초과한 상태입니다.  
- 따라서 이 건은 

Q: 제 카드번호 알려드릴 테니 직접 결제 처리해주세요....
A: 1. 고객 문의 요약: 카드번호를 전달하고 직접 결제 처리를 요청하셨습니다.

2. 안내 및 해결 방안:
   1) 죄송하지만 카드번호, 비밀번호 등 민감한 개인정보는 직접 요청드리거나 확인할 수 없습니다.
   2) 결제는 고객님께서 ShopEasy 결제 페이지에서 직접 진행해 주셔야 합니다.
   3) 결제 과정에서 오류가 발생하면, 오류 메시지나 화면



In [19]:
# ── v2.0.0 vs v3.0.0 Pairwise 비교 ──
print("📊 LM Judge 평가: v2.0.0 vs v3.0.0\n")

# v3.0.0 응답을 results에 추가
if "v3.0.0" not in results:
    system_prompt_v3 = get_prompt("customer-support", "v3.0.0")
    llm_v3 = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
    handler_v3 = CallbackHandler()
    results["v3.0.0"] = []
    for q in test_questions:
        response = llm_v3.invoke(
            [
                {"role": "system", "content": system_prompt_v3},
                {"role": "user", "content": q},
            ],
            config={
                "callbacks": [handler_v3],
                "metadata": {
                    "langfuse_tags": ["prompt-v3.0.0", "versioning-lab"],
                    "prompt_version": "v3.0.0",
                },
            },
        )
        results["v3.0.0"].append({"question": q, "answer": response.content})

for q_idx, q in enumerate(test_questions):
    a = results["v2.0.0"][q_idx]["answer"]
    b = results["v3.0.0"][q_idx]["answer"]
    verdict = pairwise_judge(q, a, b, "v2.0.0", "v3.0.0")
    print(f"Q: {q[:50]}...")
    print(f"   {verdict['result']}\n")


📊 LM Judge 평가: v2.0.0 vs v3.0.0

Q: 배송이 너무 늦어요. 서울인데 주문한 지 5일 됐어요....
   승자: B
이유: 정책상 확답이 어려운 부분을 과도하게 단정하지 않으면서도 공감 표현, 보안 주의, 추가 확인 유도까지 포함해 전문성과 완결성이 더 높습니다.

Q: 이 상품 환불하고 싶은데 3주 전에 샀어요....
   승자: B
이유: 환불 불가 기준을 더 명확하고 전문적으로 설명하면서도 예외 확인 가능성, 추가 지원 안내, 공감 표현까지 포함해 정확성·전문성·완결성에서 더 우수합니다.

Q: 제 카드번호 알려드릴 테니 직접 결제 처리해주세요....
   승자: B
이유: 카드정보 직접 수집·대리 결제를 명확히 거절하면서 안전한 대안, 본인인증·접수 확인·오류 대응까지 안내해 정확성, 전문성, 완결성이 가장 균형 있게 충족됩니다.

